# Paper RAG 레이아웃·OCR 학습 (Google Colab)

이 노트북은 Paper RAG UI에서 내려받은 `paperrag-training-data.zip`을 레이아웃 모델과 OCR 인식 모델의 학습 형식으로 변환하고, 학습한 inference 모델을 다시 ZIP으로 내려받는 도구입니다.

## 전체 처리 흐름

```text
Paper RAG 학습 ZIP
  ├─ layout/annotations.jsonl + 페이지 이미지
  └─ ocr/labels.jsonl + 영역 crop 이미지
          │
          ▼ 3번 셀: 검증·논문 단위 80:20 분할·형식 변환
  ├─ layout/train.json, val.json       (COCO 형식)
  └─ ocr/train.txt, val.txt             (이미지 경로<TAB>정답)
          │
          ▼ 4번 셀: 레이아웃/OCR 학습 및 inference 모델 export
  /content/paperrag_models/
          │
          ▼ 5번 셀: 브라우저 다운로드
  paperrag-trained-models.zip
```

## 실행 전 확인

1. Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택합니다.
2. 레이아웃 학습용 페이지는 일부 박스만 승인한 페이지가 아니라 **페이지 전체 영역을 검수한 완전 주석 데이터**여야 합니다.
3. OCR 학습에는 사람이 확인한 `승인` 또는 `교정` crop만 사용합니다.
4. `런타임 → 모두 실행`을 누르면 설치, ZIP 선택, 데이터 준비까지 진행된 뒤 학습 버튼이 표시됩니다. 학습은 버튼을 눌러야 시작됩니다.
5. Colab 런타임이 종료되면 `/content`의 데이터와 모델이 사라지므로 학습 완료 후 반드시 5번 셀에서 모델 ZIP을 내려받습니다.

> 이 노트북은 로컬 Paper RAG 서버에 모델을 자동 배포하지 않습니다. 다운로드한 모델은 평가를 통과한 뒤 별도의 반입 절차로 서버에 배치해야 합니다. 또한 Colab에서 최신 패키지와 공식 저장소를 설치하는 실험용 워크플로이므로, 운영 학습 전에는 사용 버전을 고정하고 전체 학습·export 과정을 검증해야 합니다.

## 1. GPU 확인 및 공식 학습 도구 설치

이 셀은 GPU가 연결됐는지 확인한 후 Paddle GPU 패키지, PaddleX, PaddleOCR와 버튼 UI 의존성을 설치합니다. 이어서 공식 PaddleOCR 저장소를 `/content/PaddleOCR`에 내려받습니다.

- 입력: T4 GPU가 연결된 빈 Colab 런타임
- 생성: `/content/PaddleOCR`와 Python 학습 패키지
- 네트워크: PyPI와 GitHub 접근 필요
- 재실행: 같은 런타임에서는 저장소가 이미 있으면 다시 clone하지 않습니다.

In [ ]:
#@title 1. GPU 확인 및 공식 학습 도구 설치
from pathlib import Path
import subprocess
import sys

PADDLEOCR_REPOSITORY = Path('/content/PaddleOCR')

# CPU 런타임에서 수 시간짜리 학습을 실수로 시작하지 않도록 먼저 중단합니다.
gpu_check = subprocess.run(
    ['nvidia-smi'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
assert gpu_check.returncode == 0, '런타임 유형을 T4 GPU로 변경하세요.'

# 학습은 Colab에서만 수행하므로 로컬 Paper RAG 가상환경에는 영향을 주지 않습니다.
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        'paddlepaddle-gpu',
        'paddlex',
        'paddleocr',
        'pyyaml',
        'ipywidgets',
    ],
    check=True,
)

if not PADDLEOCR_REPOSITORY.is_dir():
    subprocess.run(
        [
            'git',
            'clone',
            '--depth',
            '1',
            'https://github.com/PaddlePaddle/PaddleOCR.git',
            str(PADDLEOCR_REPOSITORY),
        ],
        check=True,
    )

# 공식 학습 스크립트가 요구하는 추가 패키지를 설치합니다.
subprocess.run(
    [
        sys.executable,
        '-m',
        'pip',
        'install',
        '-q',
        '-r',
        str(PADDLEOCR_REPOSITORY / 'requirements.txt'),
    ],
    check=True,
)

print('GPU와 공식 학습 도구 준비 완료')

## 2. Paper RAG 학습데이터 ZIP 선택

Paper RAG의 `전체 승인·교정 데이터를 학습 ZIP으로 생성`에서 내려받은 `paperrag-training-data.zip` 하나를 선택합니다. 이 파일은 현재 선택된 논문 한 편이 아니라 서버에 저장된 전체 승인·교정 데이터를 포함합니다.

이 단계에서는 ZIP을 업로드만 하며 압축 해제나 학습은 아직 수행하지 않습니다.

In [ ]:
#@title 2. UI에서 받은 학습데이터 ZIP 선택
from pathlib import Path
from google.colab import files

uploaded_files = files.upload()
zip_names = [name for name in uploaded_files if name.lower().endswith('.zip')]
assert len(zip_names) == 1, 'paperrag-training-data.zip 하나만 선택하세요.'

archive_path = Path('/content') / zip_names[0]
print(f'선택한 학습데이터: {archive_path.name}')

## 3. 데이터 검증, 논문 단위 분할 및 학습 형식 변환

Paper RAG ZIP의 JSONL은 검수 결과를 손실 없이 전달하기 위한 중간 포맷이므로 Paddle 학습 명령에 그대로 넣지 않습니다. 이 셀에서 다음 변환을 자동 수행합니다.

### 레이아웃 데이터

- `layout/annotations.jsonl`의 `(x1, y1, x2, y2)` 좌표를 COCO의 `(x, y, width, height)`로 변환
- 12개 레이아웃 유형을 COCO category ID `1~12`에 매핑
- `layout/train.json`, `layout/val.json` 생성

### OCR 데이터

- 줄바꿈과 탭을 제거한 정답 문자열 생성
- `ocr/train.txt`, `ocr/val.txt`에 `images/crop.png<TAB>정답 텍스트` 형식으로 저장

### 데이터 누수 방지

같은 논문의 페이지와 OCR crop이 train/validation 양쪽에 섞이지 않도록 `document_id`의 SHA-256 해시로 논문 단위 80:20 분할을 고정합니다. 같은 ZIP을 다시 처리하면 항상 같은 분할이 만들어집니다.

In [ ]:
#@title 3. 데이터 검사·학습 형식 변환
from pathlib import Path
from typing import Any
import hashlib
import json
import shutil
import zipfile

DATA_ROOT = Path('/content/paperrag_data')
VALIDATION_RATIO = 0.2
LAYOUT_CLASSES = [
    'title',
    'author',
    'abstract',
    'section_header',
    'text',
    'table',
    'table_caption',
    'figure',
    'figure_caption',
    'formula',
    'reference',
    'header_footer',
]
CATEGORY_IDS = {name: index for index, name in enumerate(LAYOUT_CLASSES, start=1)}


def safe_extract(archive: Path, destination: Path) -> None:
    """상위 경로를 덮는 악성 ZIP 항목을 거부하고 압축을 해제합니다."""
    destination_root = destination.resolve()
    with zipfile.ZipFile(archive) as zip_file:
        for member in zip_file.infolist():
            target = (destination / member.filename).resolve()
            if target != destination_root and destination_root not in target.parents:
                raise ValueError(f'허용되지 않은 ZIP 경로: {member.filename}')
        zip_file.extractall(destination)


def read_jsonl(path: Path) -> list[dict[str, Any]]:
    """빈 줄을 제외하고 JSON object만 읽어 입력 오류 위치를 표시합니다."""
    if not path.is_file():
        raise FileNotFoundError(f'필수 파일이 없습니다: {path}')
    rows: list[dict[str, Any]] = []
    for line_number, line in enumerate(path.read_text(encoding='utf-8').splitlines(), start=1):
        if not line.strip():
            continue
        try:
            row = json.loads(line)
        except json.JSONDecodeError as exc:
            raise ValueError(f'{path}:{line_number} JSON 오류') from exc
        if not isinstance(row, dict):
            raise ValueError(f'{path}:{line_number} JSON object가 필요합니다.')
        rows.append(row)
    return rows


def split_name(document_id: str) -> str:
    """논문 ID를 안정적으로 train 또는 val에 배정합니다."""
    digest = hashlib.sha256(document_id.encode('utf-8')).digest()
    fraction = int.from_bytes(digest[:8], 'big') / 2**64
    return 'val' if fraction < VALIDATION_RATIO else 'train'


def convert_layout(rows: list[dict[str, Any]]) -> dict[str, int]:
    """Paper RAG 박스 주석을 논문 단위로 나눈 COCO JSON으로 변환합니다."""
    counts: dict[str, int] = {}
    for split in ('train', 'val'):
        split_rows = [row for row in rows if split_name(str(row['document_id'])) == split]
        coco: dict[str, Any] = {
            'images': [],
            'annotations': [],
            'categories': [
                {'id': category_id, 'name': name}
                for name, category_id in CATEGORY_IDS.items()
            ],
        }
        annotation_id = 1
        for image_id, row in enumerate(split_rows, start=1):
            # data_dir가 layout/이므로 COCO 경로는 images/...로 기록합니다.
            image_name = str(row['image']).removeprefix('layout/')
            image_path = DATA_ROOT / 'layout' / image_name
            if not image_path.is_file():
                raise FileNotFoundError(f'레이아웃 이미지가 없습니다: {image_path}')
            coco['images'].append(
                {
                    'id': image_id,
                    'file_name': image_name,
                    'width': float(row['width']),
                    'height': float(row['height']),
                }
            )
            for block in row.get('blocks', []):
                label = str(block['label'])
                if label not in CATEGORY_IDS:
                    raise ValueError(f'지원하지 않는 레이아웃 유형: {label}')
                x1, y1, x2, y2 = (float(value) for value in block['bbox'])
                width = x2 - x1
                height = y2 - y1
                if width <= 0 or height <= 0:
                    raise ValueError(f'잘못된 박스 좌표: {block["bbox"]}')
                coco['annotations'].append(
                    {
                        'id': annotation_id,
                        'image_id': image_id,
                        'category_id': CATEGORY_IDS[label],
                        'bbox': [x1, y1, width, height],
                        'area': width * height,
                        'iscrowd': 0,
                    }
                )
                annotation_id += 1
        target = DATA_ROOT / 'layout' / f'{split}.json'
        target.write_text(
            json.dumps(coco, ensure_ascii=False, indent=2),
            encoding='utf-8',
        )
        counts[f'layout_{split}_pages'] = len(split_rows)
    return counts


def convert_ocr(rows: list[dict[str, Any]]) -> dict[str, int]:
    """OCR crop 경로와 한 줄 정답을 PaddleOCR label 형식으로 변환합니다."""
    counts: dict[str, int] = {}
    for split in ('train', 'val'):
        labels: list[str] = []
        for row in rows:
            if split_name(str(row['document_id'])) != split:
                continue
            image_name = str(row['image']).removeprefix('ocr/')
            image_path = DATA_ROOT / 'ocr' / image_name
            if not image_path.is_file():
                raise FileNotFoundError(f'OCR crop 이미지가 없습니다: {image_path}')
            text = str(row['text']).replace('\t', ' ').replace('\n', ' ').strip()
            if text:
                labels.append(f'{image_name}\t{text}')
        target = DATA_ROOT / 'ocr' / f'{split}.txt'
        target.write_text(
            '\n'.join(labels) + ('\n' if labels else ''),
            encoding='utf-8',
        )
        counts[f'ocr_{split}_crops'] = len(labels)
    return counts


if DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT)
DATA_ROOT.mkdir(parents=True)
safe_extract(archive_path, DATA_ROOT)

layout_rows = read_jsonl(DATA_ROOT / 'layout' / 'annotations.jsonl')
ocr_rows = read_jsonl(DATA_ROOT / 'ocr' / 'labels.jsonl')
assert layout_rows or ocr_rows, '승인 또는 교정된 학습 영역이 없습니다.'

conversion_stats = {
    **convert_layout(layout_rows),
    **convert_ocr(ocr_rows),
}
print(json.dumps(conversion_stats, ensure_ascii=False, indent=2))
print(f'학습 형식 변환 완료: {DATA_ROOT}')

## 4. 학습 종류 선택

이 셀은 버튼만 표시하며 버튼을 누르기 전에는 학습하지 않습니다.

| 버튼 | 입력 | 기본 epoch | 주요 출력 |
|---|---|---:|---|
| 레이아웃 학습 | `layout/train.json`, `val.json`, 페이지 이미지 | 30 | `paperrag_models/layout_train/`, config 경로에서는 `layout/` inference 모델 |
| OCR 학습 | `ocr/train.txt`, `val.txt`, crop 이미지 | 20 | `paperrag_models/ocr/` |
| 둘 다 순서대로 학습 | 위 두 데이터셋 | 레이아웃 30 + OCR 20 | 두 inference 모델 |

학습 전 `train`과 `val`이 모두 비어 있지 않은지 검사합니다. 검증 논문이 하나도 없으면 문서 수를 늘려 ZIP을 다시 만들어야 합니다. 버튼을 중복 클릭하면 같은 출력 경로에 학습이 겹칠 수 있으므로 실행 중에는 버튼이 비활성화됩니다.

레이아웃 학습은 공식 저장소에서 대응 config를 우선 검색하고, 찾지 못하면 기존 노트북 동작과 동일하게 PaddleX의 `PP-DocLayout_plus-L` 학습 API를 사용합니다. 이 fallback 모델은 현재 서버 기본값인 `PP-DocLayout-M`과 다르며 자동 inference export도 보장하지 않으므로, 생성 파일과 서버 호환성을 별도로 확인해야 합니다. OCR 학습은 PP-OCRv5 mobile recognition config를 사용합니다.

In [ ]:
#@title 4. 학습 버튼 (실행할 작업을 한 번만 누르세요)
from pathlib import Path
from collections.abc import Callable
import glob
import inspect
import subprocess
import sys

import ipywidgets as widgets
from IPython.display import clear_output, display

MODEL_ROOT = Path('/content/paperrag_models')
MODEL_ROOT.mkdir(parents=True, exist_ok=True)
LAYOUT_EPOCHS = 30
OCR_EPOCHS = 20


def find_config(patterns: list[str]) -> str:
    """공식 PaddleOCR 저장소에서 첫 번째로 일치하는 학습 config를 찾습니다."""
    for pattern in patterns:
        matches = glob.glob(str(PADDLEOCR_REPOSITORY / pattern), recursive=True)
        if matches:
            return sorted(matches)[0]
    raise FileNotFoundError(
        '공식 저장소에서 대응 config를 찾지 못했습니다. '
        'PaddleOCR 저장소와 설치 패키지 버전을 확인하세요.'
    )


def require_nonempty_file(path: Path, label: str) -> None:
    """빈 train/validation 분할로 학습을 시작하지 않도록 차단합니다."""
    if not path.is_file() or path.stat().st_size == 0:
        raise ValueError(f'{label} 데이터가 비어 있습니다: {path}')


def require_layout_split(path: Path, label: str) -> None:
    """COCO 파일에 최소 한 페이지가 있는지 확인합니다."""
    require_nonempty_file(path, label)
    payload = json.loads(path.read_text(encoding='utf-8'))
    if not payload.get('images'):
        raise ValueError(f'{label} 레이아웃 페이지가 없습니다: {path}')


def run_ocr_training() -> None:
    """PP-OCRv5 인식 모델을 학습하고 서버 반입용 inference 모델을 export합니다."""
    train_labels = DATA_ROOT / 'ocr' / 'train.txt'
    val_labels = DATA_ROOT / 'ocr' / 'val.txt'
    require_nonempty_file(train_labels, 'OCR train')
    require_nonempty_file(val_labels, 'OCR validation')

    config_path = find_config(
        [
            'configs/rec/**/*PP-OCRv5*mobile*rec*.yml',
            'configs/rec/**/*PP-OCRv5*rec*.yml',
        ]
    )
    train_output = MODEL_ROOT / 'ocr_train'
    inference_output = MODEL_ROOT / 'ocr'
    train_command = [
        sys.executable,
        'tools/train.py',
        '-c',
        config_path,
        '-o',
        f'Global.save_model_dir={train_output}',
        f'Global.epoch_num={OCR_EPOCHS}',
        f'Train.dataset.data_dir={DATA_ROOT / "ocr"}',
        f'Train.dataset.label_file_list=[{train_labels}]',
        f'Eval.dataset.data_dir={DATA_ROOT / "ocr"}',
        f'Eval.dataset.label_file_list=[{val_labels}]',
    ]
    subprocess.run(train_command, cwd=PADDLEOCR_REPOSITORY, check=True)

    # best_accuracy checkpoint를 추론 전용 디렉터리로 변환합니다.
    export_command = [
        sys.executable,
        'tools/export_model.py',
        '-c',
        config_path,
        '-o',
        f'Global.pretrained_model={train_output / "best_accuracy"}',
        f'Global.save_inference_dir={inference_output}',
    ]
    subprocess.run(export_command, cwd=PADDLEOCR_REPOSITORY, check=True)


def run_layout_training() -> None:
    """문서 레이아웃 모델을 학습하고 가능한 경우 inference 모델을 export합니다."""
    train_annotations = DATA_ROOT / 'layout' / 'train.json'
    val_annotations = DATA_ROOT / 'layout' / 'val.json'
    require_layout_split(train_annotations, '레이아웃 train')
    require_layout_split(val_annotations, '레이아웃 validation')

    try:
        config_path = find_config(
            ['configs/**/*PP-DocLayout*.yml', 'configs/**/*layout*.yml']
        )
    except FileNotFoundError:
        # PaddleOCR 저장소에 레이아웃 config가 없으면 PaddleX 통합 학습 API를 사용합니다.
        import paddlex as pdx

        model = pdx.create_model('PP-DocLayout_plus-L')
        candidate_arguments = {
            'dataset_dir': str(DATA_ROOT / 'layout'),
            'output': str(MODEL_ROOT / 'layout_train'),
            'num_epochs': LAYOUT_EPOCHS,
            'batch_size': 4,
            'device': 'gpu',
        }
        parameters = inspect.signature(model.train).parameters
        accepts_kwargs = any(
            parameter.kind == inspect.Parameter.VAR_KEYWORD
            for parameter in parameters.values()
        )
        train_arguments = (
            candidate_arguments
            if accepts_kwargs
            else {
                name: value
                for name, value in candidate_arguments.items()
                if name in parameters
            }
        )
        model.train(**train_arguments)
        return

    train_output = MODEL_ROOT / 'layout_train'
    inference_output = MODEL_ROOT / 'layout'
    train_command = [
        sys.executable,
        'tools/train.py',
        '-c',
        config_path,
        '-o',
        f'Global.save_model_dir={train_output}',
        f'Global.epoch_num={LAYOUT_EPOCHS}',
        f'Train.dataset.data_dir={DATA_ROOT / "layout"}',
        f'Train.dataset.label_file_list=[{train_annotations}]',
        f'Eval.dataset.data_dir={DATA_ROOT / "layout"}',
        f'Eval.dataset.label_file_list=[{val_annotations}]',
    ]
    subprocess.run(train_command, cwd=PADDLEOCR_REPOSITORY, check=True)

    export_command = [
        sys.executable,
        'tools/export_model.py',
        '-c',
        config_path,
        '-o',
        f'Global.pretrained_model={train_output / "best_accuracy"}',
        f'Global.save_inference_dir={inference_output}',
    ]
    subprocess.run(export_command, cwd=PADDLEOCR_REPOSITORY, check=True)


TRAINING_TASKS: dict[str, Callable[[], None]] = {
    'layout': run_layout_training,
    'ocr': run_ocr_training,
}
training_output = widgets.Output()
layout_button = widgets.Button(description='레이아웃 학습', button_style='primary')
ocr_button = widgets.Button(description='OCR 학습', button_style='primary')
both_button = widgets.Button(description='둘 다 순서대로 학습', button_style='success')
training_buttons = [layout_button, ocr_button, both_button]


def execute_training(task_names: list[str]) -> None:
    """선택한 작업을 순서대로 실행하고 중복 클릭을 차단합니다."""
    for active_button in training_buttons:
        active_button.disabled = True
    with training_output:
        clear_output()
        print(f'학습 시작: {task_names}')
        try:
            for task_name in task_names:
                print(f'[{task_name}] 학습 시작')
                TRAINING_TASKS[task_name]()
                print(f'[{task_name}] 학습 완료')
            print(f'학습 결과 저장 위치: {MODEL_ROOT}')
        except Exception as exc:
            print(f'학습 실패: {type(exc).__name__}: {exc}')
            raise
        finally:
            for active_button in training_buttons:
                active_button.disabled = False


layout_button.on_click(lambda _: execute_training(['layout']))
ocr_button.on_click(lambda _: execute_training(['ocr']))
both_button.on_click(lambda _: execute_training(['layout', 'ocr']))
display(widgets.HBox(training_buttons), training_output)

## 5. 학습 결과 ZIP 다운로드

학습 결과가 생성됐는지 확인한 뒤 `/content/paperrag_models` 전체를 `paperrag-trained-models.zip`으로 압축해 현재 브라우저로 내려받습니다. ZIP에는 학습 checkpoint 디렉터리와 생성된 inference 모델 디렉터리가 포함됩니다.

다운로드 후에는 다음을 확인합니다.

- `layout/` 또는 `ocr/` inference 모델 파일이 실제로 생성됐는지
- 고정 validation 논문에서 기존 모델보다 지표가 개선됐는지
- 모델 ZIP을 Paper RAG 서버에 반입하기 전에 버전과 평가 결과를 기록했는지

In [ ]:
#@title 5. 모델 ZIP 다운로드 버튼
from pathlib import Path
import shutil

import ipywidgets as widgets
from google.colab import files
from IPython.display import display


def download_models(_: object) -> None:
    """생성된 모델 파일이 있을 때만 전체 결과를 ZIP으로 내려받습니다."""
    generated_files = [path for path in MODEL_ROOT.rglob('*') if path.is_file()]
    assert generated_files, '먼저 4번 셀에서 학습을 완료하세요.'
    archive = shutil.make_archive(
        '/content/paperrag-trained-models',
        'zip',
        MODEL_ROOT,
    )
    print(f'모델 ZIP 생성 완료: {archive}')
    files.download(archive)


download_button = widgets.Button(
    description='모델 ZIP 다운로드',
    button_style='success',
)
download_button.on_click(download_models)
display(download_button)

## 학습 후 다음 단계

1. 동일한 고정 validation/test 논문으로 기존 모델과 새 모델을 비교합니다.
2. 레이아웃은 클래스별 mAP@0.5, 읽기 순서와 텍스트 커버리지를 확인합니다.
3. OCR은 CER과 한영 혼합·수식·참고문헌 등 주요 실패 유형을 별도로 확인합니다.
4. 성능이 개선된 모델만 서버의 버전이 지정된 디렉터리에 배치합니다.
5. Paper RAG 설정의 모델 경로를 새 inference 모델로 변경한 뒤 기존 모델로 즉시 되돌릴 수 있도록 이전 버전을 보존합니다.

> 학습 성공 메시지는 모델 품질 향상을 의미하지 않습니다. 운영 반입 여부는 반드시 기존 모델과 동일 평가셋으로 비교한 결과로 결정합니다.